# RESUME: WavLM Extraction Recovery

Use this notebook if your previous run crashed or stopped.
It will check for checkpoints and resume automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted!')

In [ ]:
!apt-get install -y ffmpeg
!pip install -q transformers librosa scikit-learn

import os, json, time, glob
import numpy as np
import torch, torch.nn as nn
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import librosa
from pathlib import Path

SR = 16000; BATCH = 16; SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load utterances
!wget -q -O /tmp/utterances_clean.jsonl.gz https://github.com/Das-rebel/chuck-audio-notebooks/raw/main/utterances_clean.jsonl.gz
!gunzip -f /tmp/utterances_clean.jsonl.gz
utterances = [json.loads(l) for l in open('/tmp/utterances_clean.jsonl')]
print(f'Loaded {len(utterances)} utterances')
vtt_data = {}
for u in utterances:
    vid = u['video_id']
    if vid not in vtt_data: vtt_data[vid] = []
    vtt_data[vid].append(u)
print(f'Unique videos: {len(vtt_data)}')

In [ ]:
# Check for existing checkpoints
checkpoint_files = sorted(glob.glob('/content/gdrive/MyDrive/extraction_checkpoint*.npz'))
print(f'Found {len(checkpoint_files)} checkpoint files:')
for f in checkpoint_files:
    try:
        ckpt = np.load(f)
        n = len(ckpt['labels'])
        print(f'  {f}: {n} utterances')
    except:
        print(f'  {f}: corrupted')

In [ ]:
# Download/extract audio if not present
!pip install -q gdown
!gdown 1jHy11LdU0bv8ra-Lj9xhpKT4yXE3L1Tp -O /tmp/vtt_audio_local.tar.gz 2>/dev/null || echo 'Already downloaded'
if os.path.exists('/tmp/vtt_audio_local'):
    print('Audio already extracted')
else:
    !tar -xzf /tmp/vtt_audio_local.tar.gz -C /tmp/
    print('Audio extracted!')

In [ ]:
# Build audio file map
audio_base = Path('/tmp/vtt_audio_local')
audio_files = {f.stem: str(f) for f in audio_base.glob('*') if f.suffix in ['.m4a', '.mp3', '.wav']}
print(f'Found {len(audio_files)} audio files')

In [ ]:
# Load WavLM
from transformers import WavLMModel
model = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
model = model.to(device).eval()
print('Model loaded!')

In [ ]:
def extract_prosody(audio_path, start, end, sr=SR):
    try:
        y, sr = librosa.load(audio_path, sr=sr, mono=True, offset=start, duration=end-start)
        if len(y) < sr * 0.05: return np.zeros(21, dtype=np.float32)
        feats = []
        try:
            f0, _, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
            f0v = f0[~np.isnan(f0)]
            feats.extend([np.mean(f0v), np.std(f0v), np.max(f0v), np.min(f0v), np.median(f0v)] if len(f0v)>0 else [0]*5)
        except: feats.extend([0]*5)
        rms = librosa.feature.rms(y=y)[0]
        feats.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms)])
        spec = [librosa.feature.spectral_centroid(y=y, sr=sr)[0], librosa.feature.spectral_bandwidth(y=y, sr=sr)[0], librosa.feature.spectral_flatness(y=y)[0], librosa.feature.zero_crossing_rate(y)[0]]
        feats.extend([np.mean(s[0]) for s in spec] + [np.max(spec[-1])])
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=7)
        feats.extend([np.mean(mfccs[i]) for i in range(7)])
        return np.array(feats, dtype=np.float32)
    except: return np.zeros(21, dtype=np.float32)

def extract_wavlm(audio_path, start, end):
    try:
        y, sr = librosa.load(audio_path, sr=SR, mono=True, offset=start, duration=end-start)
        if len(y) < SR * 0.05: return np.zeros(768, dtype=np.float32)
        y_t = torch.tensor(y).unsqueeze(0).to(device)
        with torch.no_grad(): emb = model(y_t).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        return emb
    except: return np.zeros(768, dtype=np.float32)

In [ ]:
# RESUME extraction with checkpoints
CHECKPOINT_FILE = '/content/gdrive/MyDrive/extraction_checkpoint.npz'

# Load existing checkpoint if available
if os.path.exists(CHECKPOINT_FILE):
    print('Loading existing checkpoint...')
    ckpt = np.load(CHECKPOINT_FILE)
    all_emb = list(ckpt['embeddings'])
    all_pro = list(ckpt['prosody'])
    all_lbl = list(ckpt['labels'])
    all_uid = list(ckpt['uids'])
    print(f'Resumed from: {len(all_emb)} utterances')
else:
    print('No checkpoint found. Starting fresh.')
    all_emb, all_pro, all_lbl, all_uid = [], [], [], []

# Find which utterances are already processed
processed_uids = set(all_uid)
processed_count = len(all_emb)

print(f'Processing {len(utterances)} utterances (skipping {processed_count} already done)...')
t0 = time.time()

for i, u in enumerate(utterances):
    uid = f"{u['video_id']}_{u['start']:.2f}_{u['end']:.2f}"
    if uid in processed_uids:
        continue
    
    vid = u['video_id']
    if vid not in audio_files:
        continue
    
    s, e, lbl = u['start'], u['end'], u['label']
    all_emb.append(extract_wavlm(audio_files[vid], s, e))
    all_pro.append(extract_prosody(audio_files[vid], s, e))
    all_lbl.append(lbl); all_uid.append(uid)
    
    if (len(all_emb)) % 5000 == 0:
        elapsed = time.time() - t0
        rate = len(all_emb) / elapsed
        eta = (len(utterances) - len(all_emb)) / rate / 60
        print(f'Checkpoint: {len(all_emb)}/{len(utterances)} ({rate:.1f}/s, ETA:{eta:.0f}min)')
        np.savez(CHECKPOINT_FILE,
            embeddings=np.array(all_emb, dtype=np.float32),
            prosody=np.array(all_pro, dtype=np.float32),
            labels=np.array(all_lbl, dtype=np.int64),
            uids=np.array(all_uid))
        print('Checkpoint saved!')

print(f'Done! {len(all_emb)} utterances in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Save final
np.savez('/content/gdrive/MyDrive/wavlm_prosody_555_final.npz',
    embeddings=np.array(all_emb, dtype=np.float32),
    prosody=np.array(all_pro, dtype=np.float32),
    labels=np.array(all_lbl, dtype=np.int64),
    uids=np.array(all_uid))
print(f'Saved final! {len(all_emb)} utterances')